# PFAS Groundwater — GNN Phase 1 (T1a, Colab)

**Task T1a**: predict the EPA-2024 regulatory PFAS exceedance (binary) from context only
(strict predictive mode — no PFAS measurement as a feature).

**This notebook is AUTONOMOUS** (CLAUDE.md §4): it bootstraps `src/` and the dataset via
`git clone` (no Google Drive), installs **PyTorch Geometric** for the Colab torch wheel,
builds the **well-level distance-capped spatial graph** (C4 inter-block cut), trains
GraphSAGE / GCN, and reports the honest triplet **(random, spatial, Δ)** vs the
non-graph WALL (RF spatial AUC ~0.60).

Graph design (see `experiments/gnn_phase1/REPORT.md`): node = well; edges = spatial k-NN
capped at ~1.5 km; geography enters ONLY via edges (lat/lon not node features, C6);
edges crossing a CV block boundary are cut per fold (C4); evaluation at the sampling
level for strict comparability with the WALL.

In [ ]:
# ============================================================
#  USER PARAMETERS
# ============================================================
SMOKE_TEST = False        # True = fast CPU sanity check; False = full GPU run

REPO_URL = "https://github.com/dnwiloic/pfas-gnn.git"
GIT_REF  = "main"        # commit/branch to clone
DATA_PATH = "data/CA-PFAS-ASGWS.parquet"

MODELS  = ["graphsage", "gcn"]
CAP_KM  = 1.5             # C4 hard distance cap (~1-2 km)
K_NN    = 8
N_BLOCKS = 8             # spatial blocks (reference) / random folds (Δ)

## Cell 1 — GPU detection & versions

In [ ]:
import sys, platform
print(f"Python  : {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
IN_COLAB = False
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    pass
print(f"IN_COLAB: {IN_COLAB}")
try:
    import torch
    print(f"torch   : {torch.__version__}  CUDA avail: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU     : {torch.cuda.get_device_name(0)}")
except ImportError:
    print("torch not yet installed (Colab base image usually ships it)")

## Cell 2 — Clone repo (code + versioned dataset), no Drive

In [ ]:
import os, subprocess, sys
REPO_DIR = "/content/pfas-gnn" if IN_COLAB else os.getcwd()
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", GIT_REF], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
assert os.path.exists(DATA_PATH), f"dataset missing at {DATA_PATH} — clone failed"
print("workdir:", os.getcwd())
print("dataset present:", os.path.exists(DATA_PATH))

## Cell 3 — Install PyTorch Geometric for the runtime's torch wheel

PyG's compiled extensions must match `torch.__version__` + CUDA tag. We read them at
runtime and install from the matching wheel index (idempotent: skipped if already importable).

In [ ]:
def ensure_pyg():
    try:
        import torch_geometric  # noqa
        print("torch_geometric already present:", torch_geometric.__version__)
        return
    except ImportError:
        pass
    import torch
    tv = torch.__version__.split("+")[0]
    cuda = torch.version.cuda
    tag = f"cu{cuda.replace('.', '')}" if (cuda and torch.cuda.is_available()) else "cpu"
    idx = f"https://data.pyg.org/whl/torch-{tv}+{tag}.html"
    print("installing PyG extensions from", idx)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch_geometric", "-f", idx], check=True)
    # scatter/sparse helpers (optional but speed conv); ignore failure on CPU
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch_scatter", "torch_sparse", "-f", idx])
    import torch_geometric  # noqa
    print("installed torch_geometric:", torch_geometric.__version__)

ensure_pyg()
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pyarrow>=14.0", "pyyaml>=6.0", "scikit-learn>=1.4"])

## Cell 4 — Integrity check: dataset shape & socle import

In [ ]:
from src import config as C
from src import data, gnn, graph as G, splits as S, targets as T
df = data.load(smoke=SMOKE_TEST, smoke_n=1500)
n_wells = df[C.WELL_ID].nunique()
print(f"rows={len(df)}  wells={n_wells}")
if not SMOKE_TEST:
    assert len(df) == 46338, f"unexpected row count {len(df)}"
    assert n_wells == 11333, f"unexpected well count {n_wells}"
print("T1a prevalence:", float(T.build_T1a(df).mean()).__round__(4))

## Cell 5 — Graph sanity: distance cap + C4 inter-block cut (0 cross-block edges)

In [ ]:
fb = S.spatial_block_folds(df, k=(3 if SMOKE_TEST else N_BLOCKS))
wg = G.build_well_graph(df, fold_block=fb, k=K_NN, cap_km=CAP_KM, cut_blocks=True)
a, b = wg.edge_index[0], wg.edge_index[1]
cross = int((wg.node_block[a] != wg.node_block[b]).sum())
print(f"nodes={wg.meta['n_nodes']}  undirected_edges={wg.meta['n_edges_undirected']}")
print(f"max edge length: {wg.edge_dist.max():.3f} km (cap {CAP_KM})")
print(f"C4: cross-block edges removed={wg.n_removed_cross_block}, remaining={cross}")
assert cross == 0, "C4 violated: edges still cross a CV block boundary"
assert wg.edge_dist.max() <= CAP_KM + 1e-6

## Cell 6 — Full CV: triplet (random, spatial, Δ) per model, vs the WALL

In [ ]:
import json, time
from pathlib import Path
OUT = Path("experiments/gnn_phase1"); OUT.mkdir(parents=True, exist_ok=True)
common = dict(k=K_NN, cap_km=CAP_KM, cut_blocks=True, encode="frequency",
              hidden=(32 if SMOKE_TEST else 64), layers=2, dropout=0.5,
              max_epochs=(40 if SMOKE_TEST else 300), patience=(15 if SMOKE_TEST else 30))
nb = 3 if SMOKE_TEST else N_BLOCKS
wall = {"RF_spatial": 0.601, "XGB_spatial": 0.588, "RF_delta": 0.297, "noise_floor": 0.03}
results, t0 = {}, time.time()
for model in MODELS:
    sp, _ = gnn.run_t1_cv(df, model_name=model, regime="spatial", n_blocks=nb, **common)
    rd, _ = gnn.run_t1_cv(df, model_name=model, regime="random", n_blocks=nb, **common)
    d = rd["auc_mean"] - sp["auc_mean"]
    results[model] = {"auc_spatial": sp["auc_mean"], "auc_spatial_std": sp["auc_std"],
                      "auc_random": rd["auc_mean"], "delta": d,
                      "per_fold_spatial": sp["per_fold_auc"]}
    beat = sp["auc_mean"] - wall["RF_spatial"]
    print(f"{model}: spatial={sp['auc_mean']:.4f}±{sp['auc_std']:.3f}  random={rd['auc_mean']:.4f}"
          f"  Δ={d:.4f}  | vs WALL(0.601): {beat:+.4f}"
          f" {'(> noise)' if beat>wall['noise_floor'] else '(within noise)'}")
out = {"task": "T1a", "smoke": SMOKE_TEST, "seed": C.SEED, "cap_km": CAP_KM,
       "k": K_NN, "n_blocks": nb, "models": results, "wall": wall,
       "elapsed_min": round((time.time()-t0)/60, 2)}
(OUT/"metrics.json").write_text(json.dumps(out, indent=2))
print("\nwrote", OUT/"metrics.json", "in", out["elapsed_min"], "min")

## Cell 7 — Persist outputs (ephemeral runtime, NO Drive)

The Colab workspace is lost on disconnect. Download an archive of `experiments/gnn_phase1/`
and/or commit it back to the repo. Nothing is written to Google Drive.

In [ ]:
import shutil
arch = shutil.make_archive("/content/gnn_phase1", "zip", "experiments/gnn_phase1")
print("archive:", arch)
if IN_COLAB:
    try:
        from google.colab import files
        files.download(arch)
    except Exception as e:
        print("manual download needed:", e)